In [30]:
import pandas as pd
import os
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv, find_dotenv
from sentence_transformers import SentenceTransformer

In [2]:
%load_ext dotenv
%dotenv

In [3]:
files = pd.read_csv("course_section_descriptions.csv", encoding="ANSI")

In [9]:
files.columns

Index(['course_id', 'course_name', 'course_slug', 'course_description',
       'course_description_short', 'course_technology', 'course_topic',
       'course_instructor_quote', 'section_id', 'section_name',
       'section_description'],
      dtype='object')

In [10]:
files["unique_id"] = files["course_id"].astype(str) + files["section_id"].astype(str) 

In [15]:
files["metadata"] = files.apply(lambda row :{
    "course_name": row["course_name"],
    "section_name": row["section_name"],
    "section_description": row["section_description"],
}, axis=1)

In [16]:
files.columns

Index(['course_id', 'course_name', 'course_slug', 'course_description',
       'course_description_short', 'course_technology', 'course_topic',
       'course_instructor_quote', 'section_id', 'section_name',
       'section_description', 'unique_id', 'metadata'],
      dtype='object')

In [20]:
def create_embeddings(row):
    combined_text = f""" {row["course_name"]} {row["course_technology"]} {row["course_description"]}
    {row["section_name"]} {row["section_description"]}
    """
    return model.encode(combined_text, show_progress_bar = False)

In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [21]:
files["embedding"] = files.apply(create_embeddings, axis=1)

In [22]:
files.columns

Index(['course_id', 'course_name', 'course_slug', 'course_description',
       'course_description_short', 'course_technology', 'course_topic',
       'course_instructor_quote', 'section_id', 'section_name',
       'section_description', 'unique_id', 'metadata', 'embedding'],
      dtype='object')

In [5]:
load_dotenv(find_dotenv(), override = True)

True

In [6]:
pc=Pinecone(api_key = os.environ.get("PINECONE_API_KEY"), environment = os.environ.get("PINECONE_ENV"))

In [8]:
index_name = "my-index-2"
dimension = 384
metric = "cosine"

In [26]:
if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f"{index_name} succesfully deleted.")
else:
     print(f"{index_name} not in index list.")

my-index-2 not in index list.


In [27]:
pc.create_index(
    name = index_name, 
    dimension = dimension, 
    metric = metric, 
    spec = ServerlessSpec(
        cloud = "aws", 
        region = "us-east-1")
    )

{
    "name": "my-index-2",
    "metric": "cosine",
    "host": "my-index-2-1bs5h5j.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "access-control-allow-origin": "*",
            "vary": "origin,access-control-request-method,access-control-request-headers",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "2025-10"

In [9]:
index = pc.Index(index_name)

In [30]:
vectors_to_upsert = [(row["unique_id"], row["embedding"].tolist(), row["metadata"]) for index,row in files.iterrows()]
index.upsert(vectors=vectors_to_upsert)
print("Data succesfully upserted to pinecone index")

Data succesfully upserted to pinecone index


In [28]:
query = "regression in python"
query_embedding = model.encode(query, show_progress_bar = False).tolist()

In [26]:
query_results = index.query(
    vector = [query_embedding],
    top_k = 12,
    include_metadata = True
)

In [23]:
score_threshold = 0.4

In [29]:
# Assuming query_results are fetched and include metadata
for match in query_results["matches"]:
    if match["score"] >= score_threshold:
        course_details = match.get('metadata', {})
        course_name = course_details.get('course_name', "N/A")
        section_name = course_details.get('section_name', "N/A")
        section_description = course_details.get('section_description', "No description available")

        print(f"Matched item ID: {match['id']}, Score: {match['score']}")
        print(f"Course: {course_name} \nSection: {section_name} \nDescription: {section_description} \n---------------------\n")

Matched item ID: 51466, Score: 0.487528831
Course: Machine Learning in Excel 
Section: Multiple Linear Regression 
Description: In section 3 you will discover multiple linear regression. We will expand on the simple linear regression techniques we covered in the previous section and discuss some practical considerations such as working with dummy variables and how to make predictions with more than one independent variable using Excel.  
---------------------

Matched item ID: 434, Score: 0.48468402
Course: Introduction to R Programming 
Section: Linear Regression Analysis in R 
Description: Regression analysis is another topic we covered earlier in our program. As with hypothesis testing, this is a great opportunity to apply the theory you have learned previously in R. 
---------------------

Matched item ID: 37369, Score: 0.482883036
Course: Machine Learning in Python 
Section: Linear Regression with sklearn 
Description: While there are many libraries that can compute a regression m